Warung Suara - Fine-tuning Whisper STT
========================================
Tahap 2 dari alur pengembangan (setelah generate_dataset_v2.py). Ini komponen
INTI yang WAJIB di-fine-tune sesuai rulebook AIC (poin 10 Ketentuan Khusus:
"Model wajib di fine tune sesuai dengan inovasi fitur per tim") - bukan cuma
inference seperti script TTS sebelumnya.

ALUR FINE-TUNING (untuk proposal - bagian Metodologi):
  1. Base model: `cahya/whisper-small-id` - dipilih karena sudah diadaptasi
     ke Bahasa Indonesia secara umum, jadi fine-tuning di sini tinggal fokus
     mengadaptasi ke DOMAIN SPESIFIK (istilah transaksi warung, nama barang
     dagangan, pola ucapan pedagang) - bukan belajar Bahasa Indonesia dari
     nol.
  2. Dataset: manifest.jsonl hasil generate_dataset_v2.py (audio TTS sintetik
     + label terstruktur), di-load dari Google Drive (di-zip lalu di-unzip
     otomatis ke disk lokal runtime Colab - lihat prepare_dataset_from_drive()).
     Kolom "text" dipakai sebagai target transkripsi.
  3. Preprocessing: audio di-resample ke 16kHz (kalau belum), diekstrak jadi
     log-mel spectrogram lewat WhisperFeatureExtractor, teks di-tokenize
     lewat WhisperTokenizer bawaan model.
  4. Training: Seq2SeqTrainer dari HuggingFace, dengan evaluasi WER (Word
     Error Rate) di tiap epoch untuk memantau apakah model membaik.
  5. Output: model + processor hasil fine-tune disimpan ke folder lokal,
     lalu dipanggil dari FastAPI service (/predict_risk sudah ada, endpoint
     STT baru perlu ditambahkan terpisah, misal /transcribe).

CATATAN PENTING:
- Butuh GPU (fine-tuning Whisper di CPU sangat tidak praktis, bisa berjam-jam
  bahkan untuk dataset kecil). Jalankan di Colab/lokal dengan GPU.
- Dataset ini SINTETIK (dari TTS), bukan rekaman suara manusia asli. Ini
  cukup untuk domain adaptation (istilah & pola kalimat), tapi kalau ada
  waktu luang, menambah beberapa rekaman suara ASLI (rekam sendiri/teman)
  akan signifikan meningkatkan generalisasi ke voice note pengguna beneran.
  Jelaskan keterbatasan ini secara jujur di proposal - itu sendiri bagian
  dari kriteria penilaian "MVP readiness" (area yang diakui masih bisa
  ditingkatkan).

Install dulu:
    pip install transformers datasets torch soundfile evaluate jiwer accelerate

In [ ]:
!pip install evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 40.4 MB/s eta 0:00:00


In [ ]:
import json
import shutil
import subprocess
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from datasets import Dataset, Audio
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate

# 1. KONFIGURASI

In [ ]:
BASE_MODEL_ID = "cahya/whisper-small-id"
LANGUAGE = "indonesian"
TASK = "transcribe"

# Sumber dataset (di-zip & diupload ke Drive setelah generate_dataset_v2.py)
DRIVE_DATASET_ZIP = Path("/content/drive/MyDrive/dataset_warung_suara.zip")
DRIVE_CLEAN_MANIFEST_CACHE = Path("/content/drive/MyDrive/manifest_clean.jsonl")
LOCAL_DATASET_DIR = Path("/content/dataset_warung_suara")

# Output model & checkpoint - disimpan ke Drive agar tidak hilang jika
# runtime Colab terputus di tengah training
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/warung_suara_whisper")
OUTPUT_MODEL_DIR = DRIVE_OUTPUT_DIR / "model_final"
CHECKPOINT_DIR = DRIVE_OUTPUT_DIR / "checkpoints"

TEST_SIZE = 0.1
SEED = 42

NUM_EPOCHS = 8
BATCH_SIZE = 16          # naik dari 4
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 1     # turun dari 4, karena batch size udah lebih besar        # effective batch size = BATCH_SIZE x GRAD_ACCUM_STEPS
LEARNING_RATE = 1e-5
WARMUP_STEPS = 50

# Evaluasi WER penuh (predict_with_generate) hanya dijalankan di akhir
# training, bukan tiap epoch, karena proses generate() cukup mahal secara
# memori untuk dataset berukuran ribuan sampel.
EVAL_DURING_TRAINING = False

# 2. LOAD DATASET DARI MANIFEST

In [ ]:
def ensure_dataset_ready():
    """
    Mount Google Drive, unzip dataset ke disk lokal runtime (jika belum ada),
    lalu siapkan manifest yang sudah divalidasi (audio corrupt difilter).

    Manifest bersih di-cache ke Drive supaya proses validasi (yang butuh
    membuka setiap file audio) hanya perlu dijalankan sekali, bukan setiap
    kali runtime Colab di-restart.
    """
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    if not (LOCAL_DATASET_DIR / "manifest.jsonl").exists():
        print(f"Unzip dataset dari {DRIVE_DATASET_ZIP} ...")
        extract_dir = Path("/content/_extract_tmp")
        shutil.rmtree(extract_dir, ignore_errors=True)
        extract_dir.mkdir()
        subprocess.run(["unzip", "-o", "-q", str(DRIVE_DATASET_ZIP), "-d", str(extract_dir)], check=True)

        found = list(extract_dir.rglob("manifest.jsonl"))
        if not found:
            raise FileNotFoundError(f"manifest.jsonl tidak ditemukan di dalam {DRIVE_DATASET_ZIP}")

        shutil.rmtree(LOCAL_DATASET_DIR, ignore_errors=True)
        shutil.move(str(found[0].parent), str(LOCAL_DATASET_DIR))
        shutil.rmtree(extract_dir, ignore_errors=True)

    local_clean = LOCAL_DATASET_DIR / "manifest_clean.jsonl"
    if DRIVE_CLEAN_MANIFEST_CACHE.exists():
        shutil.copy(DRIVE_CLEAN_MANIFEST_CACHE, local_clean)
    else:
        _validate_and_clean_manifest(local_clean)
        shutil.copy(local_clean, DRIVE_CLEAN_MANIFEST_CACHE)

    return local_clean


def _validate_and_clean_manifest(output_path: Path):
    """Verifikasi setiap file audio bisa dibuka; buang entri yang corrupt."""
    valid_rows, bad_count, total = [], 0, 0
    with open(LOCAL_DATASET_DIR / "manifest.jsonl", "r", encoding="utf-8") as f:
        for line in f:
            total += 1
            row = json.loads(line)
            audio_path = _resolve_audio_path(row["audio_path"])
            try:
                data, _ = sf.read(str(audio_path))
                if len(data) == 0:
                    raise ValueError("audio kosong")
                valid_rows.append(row)
            except Exception:
                bad_count += 1

    with open(output_path, "w", encoding="utf-8") as f:
        for row in valid_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"Validasi dataset: {len(valid_rows)}/{total} valid, {bad_count} dibuang (corrupt).")


def _resolve_audio_path(raw_path: str) -> Path:
    """Resolusi path audio relatif terhadap LOCAL_DATASET_DIR, robust terhadap
    perbedaan nama folder root antara saat generate vs saat training."""
    parts = Path(raw_path).parts
    if "audio" in parts:
        idx = parts.index("audio")
        return LOCAL_DATASET_DIR.joinpath(*parts[idx:])
    return Path(raw_path)


def load_manifest_dataset(manifest_path: Path):
    rows = []
    with open(manifest_path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            rows.append({"audio": str(_resolve_audio_path(row["audio_path"])), "text": row["text"]})

    dataset = Dataset.from_list(rows)
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
    dataset = dataset.train_test_split(test_size=TEST_SIZE, seed=SEED)
    print(f"Train: {len(dataset['train'])} sampel | Eval: {len(dataset['test'])} sampel")
    return dataset

# 3. PREPROCESSING (feature extraction + tokenization)

In [ ]:
def prepare_dataset(batch, feature_extractor, tokenizer):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = tokenizer(batch["text"]).input_ids
    return batch


class DataCollatorSpeechSeq2SeqWithPadding:
    """Data collator speech seq2seq: padding input_features dan labels secara terpisah."""

    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

# 4. METRIK EVALUASI (WER)

In [ ]:
def make_compute_metrics(tokenizer):
    wer_metric = evaluate.load("wer")

    def compute_metrics(pred):
        pred_ids = pred.predictions
        label_ids = pred.label_ids
        label_ids[label_ids == -100] = tokenizer.pad_token_id

        pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        return {"wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

    return compute_metrics


# 5. MAIN TRAINING PIPELINE

In [ ]:
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    if device == "cpu":
        print("WARNING: fine-tuning di CPU akan sangat lambat; disarankan pakai runtime GPU.")

    manifest_path = ensure_dataset_ready()
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Loading processor & model dari {BASE_MODEL_ID} ...")
    feature_extractor = WhisperFeatureExtractor.from_pretrained(BASE_MODEL_ID)
    tokenizer = WhisperTokenizer.from_pretrained(BASE_MODEL_ID, language=LANGUAGE, task=TASK)
    processor = WhisperProcessor.from_pretrained(BASE_MODEL_ID, language=LANGUAGE, task=TASK)
    model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL_ID, attn_implementation="sdpa")
    model.generation_config.language = LANGUAGE
    model.generation_config.task = TASK
    model.generation_config.forced_decoder_ids = None

    dataset = load_manifest_dataset(manifest_path)

    print("Preprocessing dataset (feature extraction + tokenization) ...")
    dataset = dataset.map(
        lambda batch: prepare_dataset(batch, feature_extractor, tokenizer),
        remove_columns=dataset["train"].column_names,
        num_proc=1,
        writer_batch_size=100,
    )

    data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)
    compute_metrics = make_compute_metrics(tokenizer)

    training_args = Seq2SeqTrainingArguments(
        output_dir=str(CHECKPOINT_DIR),
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=NUM_EPOCHS,
        gradient_accumulation_steps=GRAD_ACCUM_STEPS,
        gradient_checkpointing=False,
        fp16=(device == "cuda"),
        eval_strategy=("epoch" if EVAL_DURING_TRAINING else "no"),
        save_strategy="epoch",
        save_total_limit=2,
        predict_with_generate=EVAL_DURING_TRAINING,
        generation_max_length=128,
        logging_steps=25,
        load_best_model_at_end=EVAL_DURING_TRAINING,
        metric_for_best_model=("wer" if EVAL_DURING_TRAINING else None),
        greater_is_better=False,
        report_to=[],
        dataloader_num_workers=2,
    )

    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        processing_class=processor.feature_extractor,
    )

    print("Mulai training ...")
    trainer.train()

    print("Evaluasi akhir (WER) ...")
    trainer.args.predict_with_generate = True
    metrics = trainer.evaluate()
    print(f"WER akhir di eval set: {metrics['eval_wer']:.2f}%")

    print(f"Menyimpan model final ke {OUTPUT_MODEL_DIR} ...")
    trainer.save_model(str(OUTPUT_MODEL_DIR))
    processor.save_pretrained(str(OUTPUT_MODEL_DIR))
    print(f"Selesai. Model fine-tuned tersimpan di '{OUTPUT_MODEL_DIR}'.")


if __name__ == "__main__":
    main()

Device: cpu
Mounted at /content/drive
Unzip dataset dari /content/drive/MyDrive/dataset_warung_suara.zip ...
Loading processor & model dari cahya/whisper-small-id ...


preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Train: 5040 sampel | Eval: 560 sampel
Preprocessing dataset (feature extraction + tokenization) ...


Map:   0%|          | 0/5040 [00:00<?, ? examples/s]

KeyboardInterrupt: 

In [ ]:
"""
Evaluasi WER dari checkpoint terakhir (epoch 2) dan simpan sebagai model final.
Dijalankan SETELAH training di-interrupt lebih awal (loss sudah flat sejak
epoch 1, sisa epoch berpotensi cuma menambah overfitting - lihat diskusi
sebelumnya).

Jalankan di sesi Colab yang sama (kernel belum di-restart) setelah training
di-interrupt, ATAU di sesi baru asal path checkpoint di bawah sudah benar.
"""

import json
from pathlib import Path

import torch
from datasets import Dataset, Audio
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    GenerationConfig,
)
import evaluate

BASE_MODEL_ID = "cahya/whisper-small-id"
LANGUAGE = "indonesian"
TASK = "transcribe"

LOCAL_DATASET_DIR = Path("/content/dataset_warung_suara")
MANIFEST_PATH = LOCAL_DATASET_DIR / "manifest_clean.jsonl"

DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/warung_suara_whisper")
CHECKPOINT_DIR = DRIVE_OUTPUT_DIR / "checkpoints"
OUTPUT_MODEL_DIR = DRIVE_OUTPUT_DIR / "model_final"

TEST_SIZE = 0.1
SEED = 42
EVAL_BATCH_SIZE = 8


def find_latest_checkpoint():
    """Cari folder checkpoint-XXXX dengan step tertinggi di CHECKPOINT_DIR."""
    checkpoints = sorted(
        CHECKPOINT_DIR.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1]),
    )
    if not checkpoints:
        raise FileNotFoundError(f"Tidak ada checkpoint di {CHECKPOINT_DIR}")
    latest = checkpoints[-1]
    print(f"Checkpoint terakhir ditemukan: {latest}")
    return latest


def resolve_audio_path(raw_path: str) -> Path:
    parts = Path(raw_path).parts
    if "audio" in parts:
        idx = parts.index("audio")
        return LOCAL_DATASET_DIR.joinpath(*parts[idx:])
    return Path(raw_path)


def prepare_dataset(batch, feature_extractor, tokenizer):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = tokenizer(batch["text"]).input_ids
    return batch


class DataCollatorSpeechSeq2SeqWithPadding:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch


def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    checkpoint_path = find_latest_checkpoint()

    print("Loading processor & model dari checkpoint ...")
    feature_extractor = WhisperFeatureExtractor.from_pretrained(BASE_MODEL_ID)
    tokenizer = WhisperTokenizer.from_pretrained(BASE_MODEL_ID, language=LANGUAGE, task=TASK)
    processor = WhisperProcessor.from_pretrained(BASE_MODEL_ID, language=LANGUAGE, task=TASK)
    model = WhisperForConditionalGeneration.from_pretrained(str(checkpoint_path))
    # generation_config dari folder checkpoint sering tidak lengkap (tidak
    # punya lang_to_id). Repo cahya/whisper-small-id sendiri juga tidak
    # menyertakan generation_config.json, jadi diambil dari openai/whisper-small
    # (base model multilingual aslinya) yang punya mapping bahasa lengkap.
    try:
        model.generation_config = GenerationConfig.from_pretrained(BASE_MODEL_ID)
    except OSError:
        model.generation_config = GenerationConfig.from_pretrained("openai/whisper-small")
    model.generation_config.language = LANGUAGE
    model.generation_config.task = TASK
    model.generation_config.forced_decoder_ids = None

    print("Loading & preprocessing eval dataset ...")
    rows = []
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            rows.append({"audio": str(resolve_audio_path(row["audio_path"])), "text": row["text"]})

    dataset = Dataset.from_list(rows)
    dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))
    dataset = dataset.train_test_split(test_size=TEST_SIZE, seed=SEED)
    eval_dataset = dataset["test"].map(
        lambda batch: prepare_dataset(batch, feature_extractor, tokenizer),
        remove_columns=dataset["test"].column_names,
        num_proc=1,
    )

    data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)
    wer_metric = evaluate.load("wer")

    def compute_metrics(pred):
        pred_ids = pred.predictions
        label_ids = pred.label_ids
        label_ids[label_ids == -100] = tokenizer.pad_token_id
        pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
        return {"wer": 100 * wer_metric.compute(predictions=pred_str, references=label_str)}

    args = Seq2SeqTrainingArguments(
        output_dir="/content/eval_tmp",
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        predict_with_generate=True,
        generation_max_length=128,
        fp16=(device == "cuda"),
        report_to=[],
    )

    trainer = Seq2SeqTrainer(
        args=args,
        model=model,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        processing_class=processor.feature_extractor,
    )

    print("Menjalankan evaluasi WER ...")
    metrics = trainer.evaluate()
    print(f"\nWER dari checkpoint {checkpoint_path.name}: {metrics['eval_wer']:.2f}%")

    print(f"Menyimpan sebagai model final ke {OUTPUT_MODEL_DIR} ...")
    OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(OUTPUT_MODEL_DIR))
    processor.save_pretrained(str(OUTPUT_MODEL_DIR))
    print("Selesai.")


if __name__ == "__main__":
    main()

Checkpoint terakhir ditemukan: /content/drive/MyDrive/warung_suara_whisper/checkpoints/checkpoint-630
Loading processor & model dari checkpoint ...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

Loading & preprocessing eval dataset ...


Map:   0%|          | 0/560 [00:00<?, ? examples/s]

...

[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The cu

Training Loss,Validation Loss,Step,Wer
No log,0.005702,0,16.152552



WER dari checkpoint checkpoint-630: 16.15%
Menyimpan sebagai model final ke /content/drive/MyDrive/warung_suara_whisper/model_final ...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Selesai.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install huggingface_hub -q

from huggingface_hub import login
login()  # paste token HF kamu (buat di huggingface.co/settings/tokens, permission "write")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

MODEL_PATH = "/content/drive/MyDrive/warung_suara_whisper/model_final"
REPO_NAME = "IcedB/warung-suara-whisper-id"

model = WhisperForConditionalGeneration.from_pretrained(MODEL_PATH)
processor = WhisperProcessor.from_pretrained(MODEL_PATH)

model.push_to_hub(REPO_NAME)
processor.push_to_hub(REPO_NAME)

print(f"Selesai. Model tersedia di: https://huggingface.co/{REPO_NAME}")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...rrf11xu/model.safetensors:   0%|          |  575kB /  967MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Selesai. Model tersedia di: https://huggingface.co/IcedB/warung-suara-whisper-id
